# Baseline Model

Simple Random Forest baseline trained on preprocessed data.

Data:
- input: features vector
- target: label (0 = legit, 1 = fraud)

Goal:
- verify training pipeline
- generate initial predictions

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("BaselineModel") \
    .master("local[*]") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 15:30:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/13 15:30:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Training

Random Forest is used as a baseline model.
No tuning is applied at this stage.

In [2]:
train_df = spark.read.parquet("../data/processed/train")
test_df = spark.read.parquet("../data/processed/test")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20,
    maxDepth=5,
    seed=42
)

pipeline = Pipeline(stages=[rf])

model = pipeline.fit(train_df)

print("Model trained successfully")

predictions = model.transform(train_df)
predictions.select("label", "prediction").show(5)

26/05/13 15:30:41 WARN MemoryStore: Not enough space to cache rdd_32_10 in memory! (computed 211.0 MiB so far)
26/05/13 15:30:41 WARN BlockManager: Persisting block rdd_32_10 to disk instead.
26/05/13 15:30:41 WARN MemoryStore: Not enough space to cache rdd_32_3 in memory! (computed 211.0 MiB so far)
26/05/13 15:30:41 WARN BlockManager: Persisting block rdd_32_3 to disk instead.
26/05/13 15:30:46 WARN MemoryStore: Not enough space to cache rdd_32_10 in memory! (computed 211.0 MiB so far)
26/05/13 15:30:47 WARN MemoryStore: Not enough space to cache rdd_32_3 in memory! (computed 211.0 MiB so far)
26/05/13 15:30:50 WARN MemoryStore: Not enough space to cache rdd_32_3 in memory! (computed 211.0 MiB so far)
26/05/13 15:30:50 WARN MemoryStore: Not enough space to cache rdd_32_10 in memory! (computed 211.0 MiB so far)
26/05/13 15:30:55 WARN MemoryStore: Not enough space to cache rdd_32_10 in memory! (computed 211.0 MiB so far)
26/05/13 15:30:55 WARN MemoryStore: Not enough space to cache rdd

Model trained successfully


[Stage 21:===========================================>              (3 + 1) / 4]

+-----+----------+
|label|prediction|
+-----+----------+
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
+-----+----------+
only showing top 5 rows



In [3]:
predictions.groupBy("prediction").count().show()

[Stage 22:=================================================>      (14 + 2) / 16]

+----------+-------+
|prediction|  count|
+----------+-------+
|       0.0|5081354|
|       1.0|   6556|
+----------+-------+



In [4]:
predictions.groupBy("label", "prediction").count().show()

[Stage 25:=================================================>      (14 + 2) / 16]

+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0|5081323|
|    0|       1.0|      4|
|    1|       0.0|     31|
|    1|       1.0|   6552|
+-----+----------+-------+



In [5]:
print("Train rows:", train_df.count())

Train rows: 5087910


## Notes

Model produces predictions for both classes.

Results look very strong, which may indicate:
- strong patterns in data
- possible data leakage

Further evaluation is required.